# SHIVA-LLM — Objective 02 / Stage 2B
## Strict event alignment, split reconciliation and temporal leakage audit
**Runtime:** Python 3 → **CPU** (no accelerator). Estimated RAM: 4–8 GB.

**Scientific guardrail:** HyperKT export keys `(split, window_index, position)` are *not* original ASSISTments event IDs. A join based on user, skill and correctness alone is not an authenticated event mapping. This notebook **does not fabricate** an alignment. It produces a diagnostic ZIP and, only if a trustworthy event crosswalk is provided, a validated merged dataset.

**Required uploads:** `objective02_starter_results.zip`, `objective02_stage2_hyperkt_states.zip`. **Optional:** `hyperkt_event_crosswalk.csv` with `split,window_index,position,user_id,order_id` derived from the original sequence-building pipeline, plus provenance.


In [ ]:
import os, io, json, zipfile, hashlib, pathlib, numpy as np, pandas as pd
from google.colab import files
uploaded=files.upload()
required=['objective02_starter_results.zip','objective02_stage2_hyperkt_states.zip']
# Accept browser-renamed copies such as objective02_stage2_hyperkt_states(1).zip
import re
normalized={}
for name,blob in uploaded.items():
    canonical=re.sub(r'\s*\(\d+\)(?=\.zip$)', '', name, flags=re.I)
    if canonical in required:
        normalized[canonical]=blob
missing=[x for x in required if x not in normalized]
if missing: raise FileNotFoundError(f'Missing required archives: {missing}')
uploaded.update(normalized)
OUT=pathlib.Path('/content/objective02_stage2b'); OUT.mkdir(exist_ok=True)
print('Output directory:', OUT)

In [ ]:
def read_zip_csv(blob, suffix):
    with zipfile.ZipFile(io.BytesIO(blob)) as z:
        names=[name for name in z.namelist() if name.endswith(suffix)]
        if len(names)!=1: raise ValueError(f'Expected one *{suffix}, found {names}')
        return pd.read_csv(io.BytesIO(z.read(names[0])), low_memory=False)
trans=read_zip_csv(uploaded['objective02_starter_results.zip'],'transitions.csv')
parts=[]
for split,fn in [('train','hyperkt_states_train.csv'),('val','hyperkt_states_val.csv'),('test','hyperkt_states_test.csv')]:
    x=read_zip_csv(uploaded['objective02_stage2_hyperkt_states.zip'],fn)
    assert (x['split']==split).all()
    parts.append(x)
states=pd.concat(parts,ignore_index=True)
print('Stage 1:',trans.shape, '\nStage 2:',states.shape)
print('Stage 1 columns:',trans.columns.tolist())
print('Stage 2 columns:',states.columns.tolist())

In [ ]:
def split_memberships(df):
    return {s:set(g['user_id'].astype(str)) for s,g in df.groupby('split')}
a=split_memberships(trans); h=split_memberships(states)
labels=sorted(set(a)|set(h))
rows=[]
for s1 in labels:
    for s2 in labels:
        rows.append({'stage1_split':s1,'hyperkt_split':s2,'shared_learners':len(a.get(s1,set()) & h.get(s2,set()))})
confusion=pd.DataFrame(rows)
confusion.to_csv(OUT/'learner_split_crosswalk_counts.csv',index=False)
print(confusion.pivot(index='stage1_split',columns='hyperkt_split',values='shared_learners').fillna(0).astype(int))
conflicts=[]
for s1 in labels:
 for s2 in labels:
  if s1!=s2:
   for u in sorted(a.get(s1,set()) & h.get(s2,set())):
    conflicts.append({'user_id':u,'stage1_split':s1,'hyperkt_split':s2})
conflicts=pd.DataFrame(conflicts,columns=['user_id','stage1_split','hyperkt_split'])
conflicts.to_csv(OUT/'split_conflicts.csv',index=False)
print('Conflicting learner split assignments:',len(conflicts))
print(conflicts.head(10))

In [ ]:
key=['split','window_index','position']
assert not states[key].isna().any().any(), 'Missing sequence-local keys'
assert not states.duplicated(key).any(), 'Duplicate sequence-local keys'
assert states['hyperkt_pred_correct'].between(0,1).all(), 'Invalid probabilities'
assert not trans[['user_id','order_id']].isna().any().any()
assert not trans.duplicated(['user_id','order_id']).any(), 'Duplicate source event keys'
assert not states['user_id'].isna().any()
report={'stage':'objective02_stage2b','stage1_rows':int(len(trans)), 'hyperkt_rows':int(len(states)),
 'stage1_users':int(trans.user_id.nunique()),'hyperkt_users':int(states.user_id.nunique()),
 'split_conflict_users':int(len(conflicts)), 'event_alignment_verified':False,
 'warning':'Stage 2 window_index and position do not identify original events; do not join by learner or ordinal position.',
 'hyperkt_state_semantics':'pre-response target-skill correctness probability, not full latent state',
 'crosswalk_required_columns':['split','window_index','position','user_id','order_id']}
print(json.dumps(report,indent=2))

### Optional authenticated event crosswalk
A trustworthy crosswalk must be generated from the **same Stage 2 sequence-construction run** that produced the NPZ files. Each target position must retain the original `order_id` and `user_id` (and ideally source row ID, skill mapping version, sequence hashes). If unavailable, this notebook exports an **audit-only** ZIP.

**Important:** Even with a valid crosswalk, split conflicts mean the Stage 1 learner split must be discarded for a HyperKT-aligned experiment. This notebook assigns the **HyperKT split** to any validated matched event and does not train or evaluate a policy. A subsequent stage must re-fit all behavior/reward models using training-only data and document the excluded events.


In [ ]:
crosswalk_name='hyperkt_event_crosswalk.csv'
if crosswalk_name in uploaded:
    crosswalk=pd.read_csv(io.BytesIO(uploaded[crosswalk_name]),low_memory=False)
    need=set(key+['user_id','order_id'])
    assert need.issubset(crosswalk.columns), f'Missing crosswalk fields: {need-set(crosswalk.columns)}'
    assert not crosswalk.duplicated(key).any(), 'Crosswalk sequence keys duplicated'
    assert not crosswalk.duplicated(['user_id','order_id']).any(), 'Crosswalk source event keys duplicated'
    # Crosswalk must cover all exported predictions; mismatches halt execution.
    verify=states.merge(crosswalk[key+['user_id','order_id']],on=key,how='left',validate='one_to_one',indicator=True,suffixes=('','_crosswalk'))
    assert (verify['_merge']=='both').all(), 'Crosswalk does not cover every HyperKT prediction'
    assert (verify['user_id'].astype(str)==verify['user_id_crosswalk'].astype(str)).all(), 'User mismatch: crosswalk invalid'
    assert not verify['order_id'].isna().any(), 'Crosswalk has missing order_id'
    # Do not join against Stage1 split: the split protocols are different.
    left=trans.drop(columns=['split']).rename(columns={'knowledge_state':'stage1_accuracy_proxy'})
    joined=verify.drop(columns=['_merge','user_id_crosswalk']).merge(left,on=['user_id','order_id'],how='left',validate='one_to_one',indicator=True)
    report['event_alignment_verified']=True
    report['crosswalk_matches_stage1']=int((joined['_merge']=='both').sum())
    report['crosswalk_unmatched_stage1']=int((joined['_merge']!='both').sum())
    report['matched_stage1_fraction']=float((joined['_merge']=='both').mean())
    matched=joined.loc[joined['_merge']=='both'].drop(columns=['_merge'])
    matched.to_csv(OUT/'hyperkt_aligned_transitions.csv',index=False)
    print('Authenticated matched events:',len(matched),'of',len(states))
    print('CAUTION: Independently verify source row provenance and pre-response chronology before RL.')
else:
    print('No authenticated crosswalk supplied. Safe behavior: audit-only; NO joined dataset generated.')

In [ ]:
with open(OUT/'stage2b_audit.json','w') as f: json.dump(report,f,indent=2)
with open(OUT/'README.txt','w') as f:
 f.write('Objective 02 Stage 2B audit. An authenticated crosswalk is required for event-level joining.\n')
 f.write('Stage 1 and HyperKT learner partitions differ. Do not use Stage 1 policy models as held-out HyperKT evaluation.\n')
 f.write('A successful event join alone does not establish temporal causality or causal treatment effects.\n')
zip_path='/content/objective02_stage2b_audit.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
 for p in OUT.iterdir(): z.write(p,p.name)
print('Generated:',zip_path)
files.download(zip_path)